# BasementDrugDiscovery
## Notebook 06 -- Molecular Docking (GPU / GNINA)

**What this notebook does:**
Screens all prepared FDA approved ligands against each CYP51 receptor using GNINA
(GPU-accelerated docking, run as a command-line tool on the RTX 4070).
For each docking run it records the binding affinity, the interacting residues, and all relevant
docking data directly into the existing SQLite database -- one record per ligand per target.
A checkpoint system prevents duplicate runs if the process is interrupted.

**Input:**
- Receptor PDBQT file selected via popup
- Ligand PDBQT files and bdd_molecules.db from Notebook 02
- Binding site coordinates entered manually from PyMOL

**Output:**
- docking_results table added to bdd_molecules.db
- docking_contacts table with residue contact information
- Best pose PDBQT files saved per target

---

### Cell 1 -- Load all required tools

In [1]:
import os
import json
import re
import sqlite3
import subprocess
import tkinter as tk
from tkinter import filedialog, simpledialog
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

import py3Dmol
from Bio.PDB import PDBParser

import ipywidgets as widgets
from IPython.display import display, clear_output

GNINA_BIN = '/home/sardism/gnina'

def get_tk_root():
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    return root

def browse_file(title='Select file', filetypes=None):
    if filetypes is None:
        filetypes = [('All files', '*.*')]
    root = get_tk_root()
    path = filedialog.askopenfilename(title=title, filetypes=filetypes)
    root.destroy()
    return path

def browse_directory(title='Select folder'):
    root = get_tk_root()
    path = filedialog.askdirectory(title=title)
    root.destroy()
    return path

def ask_string(title, prompt, initial=''):
    root = get_tk_root()
    result = simpledialog.askstring(title=title, prompt=prompt, initialvalue=initial, parent=root)
    root.destroy()
    return result

def ask_integer(title, prompt, initial=1):
    root = get_tk_root()
    result = simpledialog.askinteger(title=title, prompt=prompt, initialvalue=initial, parent=root)
    root.destroy()
    return result

def ask_float(title, prompt, initial=0.0):
    root = get_tk_root()
    result = simpledialog.askfloat(title=title, prompt=prompt, initialvalue=initial, parent=root)
    root.destroy()
    return result

print('All tools loaded.')
print('py3Dmol ready for binding site visualization.')
print('Popup file browsers ready.')
if Path(GNINA_BIN).exists():
    print(f'GNINA GPU docking engine found: {GNINA_BIN}')
else:
    print(f'WARNING: GNINA binary not found at {GNINA_BIN}')

All tools loaded.
py3Dmol ready for binding site visualization.
Popup file browsers ready.
GNINA GPU docking engine found: /home/sardism/gnina


### Cell 2 -- Select input files and enter binding site coordinates

Click each button in order. For the binding site coordinates, use the center of mass of the co-crystallized ligand from PyMOL:
```
print(cmd.centerofmass("sele"))
```
For the box size enter the total width in Angstroms for each dimension. A value of 30 means the search extends 15 Angstroms in each direction from the center.

In [2]:
selected = {
    'db_path': None,
    'ligand_dir': None,
    'receptor_pdbqt': None,
    'poses_dir': None
}

docking_config = {
    'target_name': None,
    'target_info': None,
    'exhaustiveness': 8,
    'n_poses': 3,
    'device': 0
}

status_out = widgets.Output()

btn_db = widgets.Button(description='Browse -- Select bdd_molecules.db',
    button_style='primary', layout=widgets.Layout(width='380px'))
btn_ligands = widgets.Button(description='Browse -- Select PDBQT ligand folder',
    button_style='primary', layout=widgets.Layout(width='380px'))
btn_receptor = widgets.Button(description='Browse -- Select receptor PDBQT file',
    button_style='primary', layout=widgets.Layout(width='380px'))
btn_poses = widgets.Button(description='Browse -- Select folder to save docking poses',
    button_style='primary', layout=widgets.Layout(width='380px'))
btn_grid = widgets.Button(description='Enter target name and binding site coordinates',
    button_style='success', layout=widgets.Layout(width='380px'))

def sel_db(b):
    with status_out:
        p = browse_file('Select bdd_molecules.db',
            [('SQLite DB', '*.db'), ('All files', '*.*')])
        if p:
            selected['db_path'] = p
            print(f'Database: {p}')

def sel_ligands(b):
    with status_out:
        p = browse_directory('Select folder containing ligand PDBQT files')
        if p:
            count = len(list(Path(p).glob('*.pdbqt')))
            selected['ligand_dir'] = p
            print(f'Ligand folder: {p}')
            print(f'PDBQT files found: {count}')

def sel_receptor(b):
    with status_out:
        p = browse_file('Select receptor PDBQT file',
            [('PDBQT files', '*.pdbqt'), ('All files', '*.*')])
        if p:
            selected['receptor_pdbqt'] = p
            print(f'Receptor PDBQT: {p}')

def sel_poses(b):
    with status_out:
        p = browse_directory('Select folder to save docking poses')
        if p:
            selected['poses_dir'] = p
            print(f'Poses folder: {p}')

def enter_grid(b):
    with status_out:
        target_name = ask_string('Target Name',
            'Enter a short name for this target (e.g. candida_albicans):',
            'candida_albicans')
        pdb_id = ask_string('PDB ID',
            'Enter the PDB ID of this structure:', '5V5Z')
        cx = ask_float('Center X', 'Binding site center X (from PyMOL):', 0.0)
        cy = ask_float('Center Y', 'Binding site center Y (from PyMOL):', 0.0)
        cz = ask_float('Center Z', 'Binding site center Z (from PyMOL):', 0.0)
        sx = ask_float('Size X', 'Docking box total width X in Angstroms (e.g. 30):', 30.0)
        sy = ask_float('Size Y', 'Docking box total width Y in Angstroms (e.g. 30):', 30.0)
        sz = ask_float('Size Z', 'Docking box total width Z in Angstroms (e.g. 30):', 30.0)

        if None not in (target_name, cx, cy, cz, sx, sy, sz):
            docking_config['target_name'] = target_name.strip().lower().replace(' ', '_')
            docking_config['target_info'] = {
                'pdb_id': pdb_id.strip().upper() if pdb_id else 'UNKNOWN',
                'receptor_pdbqt': selected['receptor_pdbqt'],
                'grid_box': {
                    'center_x': cx, 'center_y': cy, 'center_z': cz,
                    'size_x': sx, 'size_y': sy, 'size_z': sz
                }
            }
            print(f'Target: {docking_config["target_name"]} ({pdb_id})')
            print(f'Grid center: ({cx}, {cy}, {cz})')
            print(f'Grid size: ({sx}, {sy}, {sz}) Angstroms')
            print('Ready. Run Cell 3 to visualize the binding site.')

btn_db.on_click(sel_db)
btn_ligands.on_click(sel_ligands)
btn_receptor.on_click(sel_receptor)
btn_poses.on_click(sel_poses)
btn_grid.on_click(enter_grid)

print('Select all inputs in order:')
display(btn_db)
display(btn_ligands)
display(btn_receptor)
display(btn_poses)
display(btn_grid)
display(status_out)

Select all inputs in order:


Button(button_style='primary', description='Browse -- Select bdd_molecules.db', layout=Layout(width='380px'), …

Button(button_style='primary', description='Browse -- Select PDBQT ligand folder', layout=Layout(width='380px'…

Button(button_style='primary', description='Browse -- Select receptor PDBQT file', layout=Layout(width='380px'…

Button(button_style='primary', description='Browse -- Select folder to save docking poses', layout=Layout(widt…

Button(button_style='success', description='Enter target name and binding site coordinates', layout=Layout(wid…

Output()

### Cell 3 -- Visualize binding site on receptor

Renders the receptor structure in 3D inside the notebook with the docking search volume shown as a red sphere. Use this to confirm the binding site coordinates are correct before starting the docking run.

In [3]:
viz_out = widgets.Output()
btn_viz = widgets.Button(
    description='Visualize binding site on receptor',
    button_style='warning',
    layout=widgets.Layout(width='350px')
)

def visualize_binding_site(b):
    with viz_out:
        clear_output()

        if not selected['receptor_pdbqt']:
            print('No receptor selected. Run Cell 2 first.')
            return
        if not docking_config['target_info']:
            print('No binding site defined. Run Cell 2 first.')
            return

        grid = docking_config['target_info']['grid_box']
        cx = grid['center_x']
        cy = grid['center_y']
        cz = grid['center_z']
        sx = grid['size_x']
        sy = grid['size_y']
        sz = grid['size_z']

        with open(selected['receptor_pdbqt']) as f:
            pdbqt_str = f.read()

        print(f'Receptor: {Path(selected["receptor_pdbqt"]).name}')
        print(f'Target: {docking_config["target_name"]} ({docking_config["target_info"]["pdb_id"]})')
        print(f'Binding site center: ({cx}, {cy}, {cz})')
        print(f'Search box: {sx} x {sy} x {sz} Angstroms')
        print('Red wireframe box = docking search volume.')
        print('Small solid red sphere = exact center.')
        print('Rotate with left click. Zoom with scroll. If the box sits over the active site the coordinates are correct.')

        view = py3Dmol.view(width=800, height=600)

        # Receptor as an opaque cartoon. A full-protein solvent-accessible
        # surface was tried here originally, but addSurface() runs
        # asynchronously (it returns a JS Promise) while every other command
        # in this view is synchronous and gets baked into a single render()
        # call by py3Dmol -- for a ~4000 atom receptor the surface resolves
        # after that render, which is what caused the model to flash and
        # then vanish. Sticking to synchronous shapes/styles only avoids it.
        view.addModel(pdbqt_str, 'pdbqt')
        view.setStyle({'model': 0}, {
            'cartoon': {'color': 'lightgray', 'opacity': 1.0}
        })

        # Wireframe box showing the actual GNINA search volume (a box, not
        # a sphere -- this also matches the real grid_box dimensions instead
        # of an averaged-radius approximation).
        view.addBox({
            'center': {'x': cx, 'y': cy, 'z': cz},
            'dimensions': {'w': sx, 'h': sy, 'd': sz},
            'color': 'red',
            'wireframe': True,
            'linewidth': 3.0
        })

        # Small solid sphere at the exact center.
        view.addSphere({
            'center': {'x': cx, 'y': cy, 'z': cz},
            'radius': 1.5,
            'color': 'red',
            'opacity': 1.0
        })

        view.zoomTo()
        view.show()

btn_viz.on_click(visualize_binding_site)
print('Click to visualize the binding site:')
display(btn_viz)
display(viz_out)

Click to visualize the binding site:


Button(button_style='warning', description='Visualize binding site on receptor', layout=Layout(width='350px'),…

Output()

### Cell 4 -- Set docking parameters

In [4]:
params_out = widgets.Output()
btn_params = widgets.Button(
    description='Set docking parameters',
    button_style='info',
    layout=widgets.Layout(width='350px')
)

def set_params(b):
    with params_out:
        clear_output()
        exhaustiveness = ask_integer('Exhaustiveness',
            'Docking exhaustiveness (8=standard, 16=thorough, 32=very thorough):', 8)
        n_poses = ask_integer('Number of Poses',
            'Number of poses to generate per ligand (1-9):', 3)
        device = ask_integer('GPU Device',
            'GPU device index for GNINA to use (0 = first GPU):', 0)
        if exhaustiveness: docking_config['exhaustiveness'] = exhaustiveness
        if n_poses: docking_config['n_poses'] = n_poses
        if device is not None: docking_config['device'] = device
        print(f'Parameters set:')
        print(f'  Exhaustiveness: {docking_config["exhaustiveness"]}')
        print(f'  Poses per ligand: {docking_config["n_poses"]}')
        print(f'  GPU device: {docking_config["device"]}')

btn_params.on_click(set_params)
print('Optional -- adjust docking parameters (defaults are fine for most runs):')
display(btn_params)
display(params_out)

Optional -- adjust docking parameters (defaults are fine for most runs):


Button(button_style='info', description='Set docking parameters', layout=Layout(width='350px'), style=ButtonSt…

Output()

### Cell 5 -- Initialize database tables for docking results

Adds docking_results and docking_contacts tables to the existing bdd_molecules.db.
Safe to run multiple times -- existing data is never overwritten.

In [5]:
def initialize_docking_tables(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS docking_results (
            result_id           INTEGER PRIMARY KEY AUTOINCREMENT,
            chembl_id           TEXT,
            target_name         TEXT,
            pdb_id              TEXT,
            affinity_best       REAL,
            affinity_2nd        REAL,
            affinity_3rd        REAL,
            n_poses             INTEGER,
            pose_file           TEXT,
            exhaustiveness      INTEGER,
            grid_center_x       REAL,
            grid_center_y       REAL,
            grid_center_z       REAL,
            grid_size_x         REAL,
            grid_size_y         REAL,
            grid_size_z         REAL,
            status              TEXT DEFAULT 'success',
            error_message       TEXT,
            timestamp           TEXT,
            UNIQUE(chembl_id, target_name)
        )
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS docking_contacts (
            contact_id          INTEGER PRIMARY KEY AUTOINCREMENT,
            result_id           INTEGER,
            chembl_id           TEXT,
            target_name         TEXT,
            residue_name        TEXT,
            residue_number      INTEGER,
            chain               TEXT,
            contact_distance    REAL,
            FOREIGN KEY (result_id) REFERENCES docking_results(result_id)
        )
    ''')

    conn.commit()

    existing = cursor.execute('SELECT COUNT(*) FROM docking_results').fetchone()[0]
    molecules = cursor.execute(
        "SELECT COUNT(*) FROM molecules WHERE status = 'success'"
    ).fetchone()[0]

    conn.close()

    print(f'Database: {db_path}')
    print(f'Tables initialized: docking_results, docking_contacts')
    print(f'Existing docking results: {existing}')
    print(f'Molecules available for docking: {molecules}')


if selected['db_path']:
    initialize_docking_tables(selected['db_path'])
else:
    print('No database selected. Run Cell 2 first.')

Database: /home/sardism/BasementDrugDiscovery/data/bdd_fdamolecules.db
Tables initialized: docking_results, docking_contacts
Existing docking results: 5728
Molecules available for docking: 2864


### Cell 6 -- Checkpoint: get pending molecules

Checks which molecules have already been docked against the selected target.
Returns only molecules still needing processing. Safe to call after any interruption.

In [6]:
def get_pending_docking(db_path, target_name, ligand_dir):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    all_ready = cursor.execute(
        "SELECT chembl_id, name FROM molecules WHERE status = 'success'"
    ).fetchall()

    already_done = set(row[0] for row in cursor.execute(
        'SELECT chembl_id FROM docking_results WHERE target_name = ?',
        (target_name,)
    ).fetchall())

    conn.close()

    ligand_dir = Path(ligand_dir)
    pending = []
    missing_pdbqt = 0

    for chembl_id, name in all_ready:
        if chembl_id in already_done:
            continue
        pdbqt_file = ligand_dir / f'{chembl_id}.pdbqt'
        if pdbqt_file.exists():
            pending.append((chembl_id, name, str(pdbqt_file)))
        else:
            missing_pdbqt += 1

    print(f'Target: {target_name}')
    print(f'Total prepared molecules: {len(all_ready)}')
    print(f'Already docked against this target: {len(already_done)}')
    print(f'Missing PDBQT files: {missing_pdbqt}')
    print(f'Pending for this run: {len(pending)}')

    return pending


if all(selected[k] for k in ['db_path', 'ligand_dir']) and docking_config['target_name']:
    pending_ligands = get_pending_docking(
        selected['db_path'],
        docking_config['target_name'],
        selected['ligand_dir']
    )
    print(f'\nReady to start docking.')
else:
    print('Complete Cell 2 first.')

Target: homo_sapienscyp51
Total prepared molecules: 2864
Already docked against this target: 0
Missing PDBQT files: 0
Pending for this run: 2864

Ready to start docking.


### Cell 7 -- Define contact analysis function

Finds receptor residues within 4.5 Angstroms of the docked ligand.
Reads directly from PDBQT files -- no separate PDB file needed.

In [7]:
CONTACT_DISTANCE = 4.5  # Angstroms
HYDROGEN_TYPES = {'H', 'HD', 'HS'}  # AutoDock atom types for hydrogen

def parse_pdbqt_coords(pdbqt_path):
    coords = []
    with open(pdbqt_path) as f:
        for line in f:
            if line.startswith(('ATOM', 'HETATM')):
                try:
                    x = float(line[30:38])
                    y = float(line[38:46])
                    z = float(line[46:54])
                    element = line[77:79].strip() if len(line) > 77 else 'X'
                    if element not in HYDROGEN_TYPES:
                        coords.append([x, y, z])
                except:
                    pass
            elif line.startswith('ENDMDL'):
                break
    return np.array(coords) if coords else np.array([])


def find_contacts(pose_pdbqt, receptor_pdbqt, cutoff=CONTACT_DISTANCE):
    ligand_coords = parse_pdbqt_coords(pose_pdbqt)
    if len(ligand_coords) == 0:
        return []

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('receptor', receptor_pdbqt)

    contacts = []
    seen_residues = set()

    for model in structure:
        for chain in model:
            for residue in chain:
                resname = residue.get_resname()
                resnum = residue.id[1]
                chain_id = chain.id
                res_key = (resname, resnum, chain_id)

                if res_key in seen_residues:
                    continue

                for atom in residue:
                    # PDBQT's AutoDock atom-type column doesn't line up with
                    # the standard PDB element column, so Bio.PDB's atom.element
                    # can't be trusted here. The atom name column is unaffected.
                    if atom.get_name().strip().upper().startswith('H'):
                        continue
                    atom_coord = atom.get_coord()
                    diffs = ligand_coords - atom_coord
                    distances = np.sqrt((diffs**2).sum(axis=1))
                    min_dist = distances.min()

                    if min_dist <= cutoff:
                        contacts.append((
                            resname, resnum, chain_id,
                            round(float(min_dist), 3)
                        ))
                        seen_residues.add(res_key)
                        break
        break

    contacts.sort(key=lambda x: x[3])
    return contacts


print('Contact analysis function defined.')
print(f'Contact cutoff: {CONTACT_DISTANCE} Angstroms')

Contact analysis function defined.
Contact cutoff: 4.5 Angstroms


### Cell 8 -- Run the docking pipeline

Docks all pending ligands against the selected target.
Updates the database after each molecule.
Safe to interrupt and restart -- Cell 6 picks up from where this stopped.

Expected runtime: a few seconds per molecule on GPU at exhaustiveness 8 (much faster than CPU Vina).

In [8]:
import time

GNINA_TIMEOUT_SECONDS = 600
AFFINITY_RE = re.compile(r'^REMARK minimizedAffinity\s+(-?\d+\.?\d*)')

def parse_gnina_affinities(pdbqt_path):
    affinities = []
    with open(pdbqt_path) as f:
        for line in f:
            m = AFFINITY_RE.match(line)
            if m:
                affinities.append(float(m.group(1)))
    return affinities


def extract_best_pose(multi_pose_pdbqt, single_pose_out):
    # GNINA writes every requested pose into one PDBQT file. Vina's API wrote
    # only the single best pose to disk, and downstream contact analysis /
    # the results tables assume one pose per file, so keep only the first
    # (best-affinity, since we sort with --pose_sort_order Energy) MODEL block.
    best_lines = []
    with open(multi_pose_pdbqt) as f:
        for line in f:
            best_lines.append(line)
            if line.startswith('ENDMDL'):
                break
    with open(single_pose_out, 'w') as f:
        f.writelines(best_lines)


def run_gnina(receptor_pdbqt, ligand_pdbqt, out_pdbqt, grid, exhaustiveness, n_poses, device):
    cmd = [
        GNINA_BIN,
        '--receptor', receptor_pdbqt,
        '--ligand', ligand_pdbqt,
        '--out', str(out_pdbqt),
        '--center_x', str(grid['center_x']),
        '--center_y', str(grid['center_y']),
        '--center_z', str(grid['center_z']),
        '--size_x', str(grid['size_x']),
        '--size_y', str(grid['size_y']),
        '--size_z', str(grid['size_z']),
        '--exhaustiveness', str(exhaustiveness),
        '--num_modes', str(n_poses),
        '--pose_sort_order', 'Energy',
        '--device', str(device),
        '--quiet'
    ]
    result = subprocess.run(
        cmd, capture_output=True, text=True, timeout=GNINA_TIMEOUT_SECONDS
    )
    if result.returncode != 0:
        raise RuntimeError(f'GNINA exited with code {result.returncode}: {result.stderr[-500:]}')
    return result


def run_docking_pipeline(pending_ligands, docking_config, selected, db_path):
    if not pending_ligands:
        print('No molecules to dock. All done for this target.')
        return

    target_name = docking_config['target_name']
    target_info = docking_config['target_info']
    grid = target_info['grid_box']
    # Read the receptor path live from `selected` (same source Cell 3 uses)
    # rather than target_info's copy -- target_info['receptor_pdbqt'] is
    # only a snapshot taken when the grid coordinates were entered in Cell 2,
    # and goes stale if the receptor file is (re)selected afterward.
    receptor_pdbqt = selected['receptor_pdbqt']
    poses_dir = Path(selected['poses_dir']) / target_name
    poses_dir.mkdir(parents=True, exist_ok=True)

    total = len(pending_ligands)
    success_count = 0
    failed_count = 0
    last_affinity = 0.0
    start_time = time.time()

    progress = widgets.IntProgress(
        value=0, min=0, max=total,
        description='Docking:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='700px')
    )
    status_label = widgets.Label(value=f'Initializing GNINA on GPU device {docking_config["device"]}...')
    display(progress)
    display(status_label)

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    for i, (chembl_id, name, ligand_pdbqt) in enumerate(pending_ligands):
        raw_pose_file = poses_dir / f'{chembl_id}_gnina_raw.pdbqt'
        pose_file = str(poses_dir / f'{chembl_id}_best_pose.pdbqt')
        try:
            run_gnina(
                receptor_pdbqt, ligand_pdbqt, raw_pose_file, grid,
                docking_config['exhaustiveness'], docking_config['n_poses'],
                docking_config['device']
            )

            affinities = parse_gnina_affinities(raw_pose_file)

            affinity_best = affinities[0] if len(affinities) > 0 else None
            affinity_2nd  = affinities[1] if len(affinities) > 1 else None
            affinity_3rd  = affinities[2] if len(affinities) > 2 else None

            if affinity_best is not None:
                last_affinity = affinity_best

            extract_best_pose(raw_pose_file, pose_file)
            raw_pose_file.unlink(missing_ok=True)

            cursor.execute('''
                INSERT OR IGNORE INTO docking_results
                (chembl_id, target_name, pdb_id, affinity_best, affinity_2nd, affinity_3rd,
                 n_poses, pose_file, exhaustiveness,
                 grid_center_x, grid_center_y, grid_center_z,
                 grid_size_x, grid_size_y, grid_size_z,
                 status, timestamp)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                chembl_id, target_name, target_info['pdb_id'],
                affinity_best, affinity_2nd, affinity_3rd,
                len(affinities), pose_file,
                docking_config['exhaustiveness'],
                grid['center_x'], grid['center_y'], grid['center_z'],
                grid['size_x'], grid['size_y'], grid['size_z'],
                'success', datetime.now().isoformat()
            ))

            result_id = cursor.lastrowid

            if Path(pose_file).exists():
                contacts = find_contacts(pose_file, receptor_pdbqt)
                for resname, resnum, chain_id, dist in contacts:
                    cursor.execute('''
                        INSERT INTO docking_contacts
                        (result_id, chembl_id, target_name,
                         residue_name, residue_number, chain, contact_distance)
                        VALUES (?, ?, ?, ?, ?, ?, ?)
                    ''', (result_id, chembl_id, target_name,
                          resname, resnum, chain_id, dist))

            success_count += 1

        except Exception as e:
            Path(raw_pose_file).unlink(missing_ok=True)
            cursor.execute('''
                INSERT OR IGNORE INTO docking_results
                (chembl_id, target_name, pdb_id, status, error_message, timestamp)
                VALUES (?, ?, ?, ?, ?, ?)
            ''', (
                chembl_id, target_name, target_info['pdb_id'],
                'failed', str(e)[:500], datetime.now().isoformat()
            ))
            failed_count += 1

        if (i + 1) % 20 == 0:
            conn.commit()

        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        remaining = (total - i - 1) / rate if rate > 0 else 0
        progress.value = i + 1
        status_label.value = (
            f'{i+1}/{total} -- '
            f'Success: {success_count} -- '
            f'Failed: {failed_count} -- '
            f'Best so far: {last_affinity:.1f} kcal/mol -- '
            f'Remaining: {remaining/60:.1f} min'
        )

    conn.commit()
    conn.close()

    elapsed_total = time.time() - start_time
    print(f'\nDocking complete for {target_name}.')
    print(f'Total docked: {total}')
    print(f'Success: {success_count} ({100*success_count/total:.1f}%)')
    print(f'Failed: {failed_count}')
    print(f'Total time: {elapsed_total/60:.1f} minutes')


ready = all([
    selected['db_path'],
    selected['ligand_dir'],
    selected['receptor_pdbqt'],
    selected['poses_dir'],
    docking_config['target_name'],
    docking_config['target_info']
])

if not ready:
    print('Not ready. Complete Cell 2 first.')
elif not pending_ligands:
    print('No pending ligands. Run Cell 6 first.')
else:
    print(f'Starting docking run.')
    print(f'Target: {docking_config["target_name"]}')
    print(f'Ligands to dock: {len(pending_ligands)}')
    print(f'Exhaustiveness: {docking_config["exhaustiveness"]}')
    print(f'Poses per ligand: {docking_config["n_poses"]}')
    print(f'GPU device: {docking_config["device"]}')
    print('\nYou can safely interrupt at any time.')
    print('Restart by running Cell 6 then this cell again.\n')
    run_docking_pipeline(pending_ligands, docking_config, selected, selected['db_path'])

Starting docking run.
Target: homo_sapienscyp51
Ligands to dock: 2864
Exhaustiveness: 8
Poses per ligand: 3
GPU device: 0

You can safely interrupt at any time.
Restart by running Cell 6 then this cell again.



IntProgress(value=0, description='Docking:', layout=Layout(width='700px'), max=2864, style=ProgressStyle(descr…

Label(value='Initializing GNINA on GPU device 0...')


Docking complete for homo_sapienscyp51.
Total docked: 2864
Success: 2714 (94.8%)
Failed: 150
Total time: 659.4 minutes


### Cell 9 -- Quick results summary

In [ ]:
if selected['db_path'] and docking_config['target_name']:
    conn = sqlite3.connect(selected['db_path'])
    target = docking_config['target_name']

    status_summary = pd.read_sql('''
        SELECT status, COUNT(*) as count
        FROM docking_results WHERE target_name = ?
        GROUP BY status
    ''', conn, params=(target,))
    print(f'Docking status for {target}:')
    display(status_summary)

    top_hits = pd.read_sql('''
        SELECT dr.chembl_id, m.name, dr.affinity_best, dr.affinity_2nd, dr.affinity_3rd,
               m.molecular_weight, m.alogp, m.qed_score, m.known_antifungal, dr.pose_file
        FROM docking_results dr
        JOIN molecules m ON dr.chembl_id = m.chembl_id
        WHERE dr.target_name = ? AND dr.status = 'success'
        ORDER BY dr.affinity_best ASC LIMIT 20
    ''', conn, params=(target,))
    print(f'\nTop 20 hits (most negative affinity = strongest binding):')
    display(top_hits)

    known = top_hits[top_hits['known_antifungal'] == 1]
    if len(known) > 0:
        print(f'\nKnown antifungals in top 20 (pipeline validation):')
        display(known[['chembl_id', 'name', 'affinity_best']])

    all_affinities = pd.read_sql('''
        SELECT affinity_best FROM docking_results
        WHERE target_name = ? AND status = 'success' AND affinity_best IS NOT NULL
    ''', conn, params=(target,))

    if len(all_affinities) > 0:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.hist(all_affinities['affinity_best'], bins=50, color='steelblue', alpha=0.7, edgecolor='white')
        ax.axvline(x=all_affinities['affinity_best'].quantile(0.05),
                   color='coral', linestyle='--', label='Top 5% cutoff')
        ax.set_xlabel('Binding Affinity (kcal/mol)', fontsize=12)
        ax.set_ylabel('Count', fontsize=12)
        ax.set_title(f'Affinity Distribution -- {target} ({len(all_affinities)} molecules)', fontsize=12)
        ax.legend()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.show()

    conn.close()
else:
    print('Complete Cell 2 first.')

### Cell 10 -- Show contacts for a specific hit

In [ ]:
contact_out = widgets.Output()
btn_contacts = widgets.Button(
    description='Enter ChEMBL ID to show contacts',
    button_style='info',
    layout=widgets.Layout(width='350px')
)

def show_contacts(b):
    with contact_out:
        clear_output()
        chembl_id = ask_string('ChEMBL ID',
            'Enter the ChEMBL ID of the compound to inspect:', initial='CHEMBL')
        if not chembl_id:
            return

        conn = sqlite3.connect(selected['db_path'])
        result = pd.read_sql('''
            SELECT dr.chembl_id, m.name, dr.affinity_best, dr.target_name, dr.pdb_id
            FROM docking_results dr
            JOIN molecules m ON dr.chembl_id = m.chembl_id
            WHERE dr.chembl_id = ? AND dr.target_name = ?
        ''', conn, params=(chembl_id, docking_config['target_name']))

        if len(result) == 0:
            print(f'No result found for {chembl_id} against {docking_config["target_name"]}')
            conn.close()
            return

        print(f'Compound: {result.iloc[0]["name"]} ({chembl_id})')
        print(f'Target: {result.iloc[0]["target_name"]} ({result.iloc[0]["pdb_id"]})')
        print(f'Best affinity: {result.iloc[0]["affinity_best"]:.2f} kcal/mol')

        contacts = pd.read_sql('''
            SELECT residue_name, residue_number, chain, contact_distance
            FROM docking_contacts
            WHERE chembl_id = ? AND target_name = ?
            ORDER BY contact_distance ASC
        ''', conn, params=(chembl_id, docking_config['target_name']))
        conn.close()

        if len(contacts) > 0:
            print(f'\nContacting residues (within {CONTACT_DISTANCE} Angstroms):')
            display(contacts)
        else:
            print('No contact data found.')

btn_contacts.on_click(show_contacts)
display(btn_contacts)
display(contact_out)

### Cell 11 -- Export top hits to CSV

In [ ]:
export_out = widgets.Output()
btn_export = widgets.Button(
    description='Export top hits to CSV',
    button_style='success',
    layout=widgets.Layout(width='280px')
)

def export_hits(b):
    with export_out:
        clear_output()
        n = ask_integer('Top N hits', 'How many top hits to export?', 100)
        if not n:
            return
        save_dir = browse_directory('Select folder to save export')
        if not save_dir:
            return

        target = docking_config['target_name']
        conn = sqlite3.connect(selected['db_path'])

        df = pd.read_sql('''
            SELECT dr.chembl_id, m.name, m.smiles,
                   dr.affinity_best, dr.affinity_2nd, dr.affinity_3rd,
                   m.molecular_weight, m.alogp, m.hbd, m.hba, m.psa,
                   m.qed_score, m.fsp3, m.lipinski_pass, m.known_antifungal,
                   dr.pose_file
            FROM docking_results dr
            JOIN molecules m ON dr.chembl_id = m.chembl_id
            WHERE dr.target_name = ? AND dr.status = 'success'
            ORDER BY dr.affinity_best ASC LIMIT ?
        ''', conn, params=(target, n))
        conn.close()

        export_path = Path(save_dir) / f'{target}_top{n}_hits.csv'
        df.to_csv(export_path, index=False)

        print(f'Exported {len(df)} hits to: {export_path}')
        print(f'Best affinity: {df["affinity_best"].min():.2f} kcal/mol')
        print(f'Worst affinity: {df["affinity_best"].max():.2f} kcal/mol')
        display(df.head(10))

btn_export.on_click(export_hits)
display(btn_export)
display(export_out)

---
## Running against multiple targets

To dock against a different target go back to Cell 2, click the receptor and coordinates buttons again with the new target details, then run Cell 3 to visualize and confirm, Cell 6 to get pending molecules, and Cell 8 to dock. The checkpoint system ensures each target accumulates results independently.

**GitHub:** https://github.com/sardism/BasementDrugDiscovery

**Note:** All results are computational predictions requiring experimental validation.